In [15]:
!pip install -q -U langgraph langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 5.2 MB/s eta 0:00:00


In [17]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

# LLM create cheyali
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

In [18]:
from langchain.tools import tool
from langchain_core.messages import ToolMessage

# Tools related
@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b


@tool
def divide(a: int, b: int) -> float:
    """Divide two numbers."""
    return a / b


tools = [add, multiply, divide]

tools_by_name = {
    tool.name: tool
    for tool in tools
}

In [19]:
model_with_tools = llm.bind_tools(tools)

In [21]:
#State
from langgraph.graph import StateGraph, START, END, MessagesState

class State(TypedDict):
    messages: list

In [22]:
#LLM Node
def llm_call(state: MessagesState):
    response = model_with_tools.invoke(state["messages"])

    return {
        "messages": [response]
    }

In [23]:
#Tool nodes
def tool_node(state: MessagesState):
    results = []

    last_message = state["messages"][-1]

    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        selected_tool = tools_by_name[tool_name]

        result = selected_tool.invoke(tool_args)

        results.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )

    return {
        "messages": results
    }

In [24]:
#Routing

def should_continue(state: MessagesState):
    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tool_node"

    return "end"

In [25]:
#Graph building

builder = StateGraph(MessagesState)

builder.add_node("llm_call", llm_call)
builder.add_node("tool_node", tool_node)

builder.add_edge(START, "llm_call")

builder.add_conditional_edges(
    "llm_call",
    should_continue,
    {
        "tool_node": "tool_node",
        "end": END
    }
)

builder.add_edge("tool_node", "llm_call")

agent = builder.compile()

In [29]:
#o/p

calculation = input("Enter your calculation: ")

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": calculation
        }
    ]
})

print(result["messages"][-1].content[0]['text'])

Enter your calculation: add 100 and -50
The sum of 100 and -50 is **50**.
